现在我们来讨论一个延迟绑定的陷阱：
这里以filter的使用为例：
给定一个数n，求n以内的所有素数：

In [ ]:
def iter():
    n=2
    while True:
        yield n
        n+=1

def isprime(n):
    it=iter()
    u=next(it)
    while u<n:
        yield u
        it=filter(lambda x,p=u:x%p>0,it)
        u=next(it)

for i in isprime(100):
    print(i)


函数iter是一个序列生成器，主要用来生成2到n这个序列。
这是一个正确的代码，现在我们来看错误的例子：

In [ ]:
def iter():
    n=2
    while True:
        yield n
        n+=1

def isprime(n):
    it=iter()
    u=next(it)
    while u<n:
        yield u
        it=filter(lambda x:x%u>0,it)
        u=next(it)

for i in isprime(100):
    print(i)


我们在lambda函数内删除了一个变量p，原本这个p是用来接收u的值。
这个输出的结果为2到n整个序列，filter完全没有起到筛选过滤的作用，为什么呢？

原因在于，filter的原理是生成一层层滤网：正常情况下是滤网f1过滤2的倍数，f2过滤3的倍数，f3过滤5的倍数...，可是u这个变量使用一直在变化啊，这样会导致你的滤网一直都是在过滤u的倍数，加上绑定延迟，导致完全没有过滤。

看到正确结果，其实还是很好看懂的，就是用一个变量p去“记住”当前的u，从而让滤网得到实现。

还有一种办法：外部定义一个函数。

In [ ]:
def iter():
    n=2
    while True:
        yield n
        n+=1
def divided(p):
    return lambda x:x%p>0
def isprime(n):
    it=iter()
    u=next(it)
    while u<n:
        yield u
        it=filter(divided(u),it)
        u=next(it)

for i in isprime(100):
    print(i)


原理是一样的，也是函数形参记住了当前的u值，滤网不会因为u的变化而变化。

目前我正在学习python，后续还有什么方法我将补充：

In [ ]:
from functools import partial

def iter():
    n=2
    while True:
        yield n
        n+=1

def isprime(n):
    it=iter()
    u=next(it)
    while u<n:
        yield u
        it=filter(partial(lambda x,p:x%p>0,p=u),it)
        u=next(it)

for i in isprime(10):
    print(i)


我现在用偏函数来做，但是我感觉有点多此一举了，目的还是让p固定了当前的u。